In [1]:
'''
Install dependencies 
pip install opencv-python 
pip install mediapipe
'''
# Modified Cell 1: support both legacy mp.solutions and the current MediaPipe Tasks API.
import os
import time

import cv2
import mediapipe as mp

USING_LEGACY_SOLUTIONS = hasattr(mp, 'solutions')

if USING_LEGACY_SOLUTIONS:
    mp_holistic = mp.solutions.holistic
    mp_drawing = mp.solutions.drawing_utils
    print('Using legacy MediaPipe solutions.holistic API')
else:
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    from mediapipe.tasks.python.vision import drawing_utils as mp_drawing

    HOLISTIC_MODEL_PATH = 'holistic_landmarker.task'
    HAND_MODEL_PATH = 'hand_landmarker.task'
    POSE_MODEL_PATH = 'pose_landmarker.task' if os.path.exists('pose_landmarker.task') else 'pose_landmarker_heavy.task'

    class CompatHolisticResult:
        def __init__(self, face_landmarks, pose_landmarks, left_hand_landmarks, right_hand_landmarks):
            self.face_landmarks = face_landmarks
            self.pose_landmarks = pose_landmarks
            self.left_hand_landmarks = left_hand_landmarks
            self.right_hand_landmarks = right_hand_landmarks

    class CombinedHolisticRunner:
        def __init__(self, pose_model_path, hand_model_path):
            self.pose_model_path = pose_model_path
            self.hand_model_path = hand_model_path
            self.pose_landmarker = None
            self.hand_landmarker = None

        def __enter__(self):
            pose_options = vision.PoseLandmarkerOptions(
                base_options=python.BaseOptions(model_asset_path=self.pose_model_path),
                running_mode=vision.RunningMode.VIDEO,
                min_pose_detection_confidence=0.5,
                min_pose_presence_confidence=0.5,
                min_tracking_confidence=0.5,
            )
            hand_options = vision.HandLandmarkerOptions(
                base_options=python.BaseOptions(model_asset_path=self.hand_model_path),
                running_mode=vision.RunningMode.VIDEO,
                num_hands=2,
                min_hand_detection_confidence=0.5,
                min_hand_presence_confidence=0.5,
                min_tracking_confidence=0.5,
            )
            self.pose_landmarker = vision.PoseLandmarker.create_from_options(pose_options)
            self.hand_landmarker = vision.HandLandmarker.create_from_options(hand_options)
            return self

        def __exit__(self, exc_type, exc_value, traceback):
            if self.pose_landmarker is not None:
                self.pose_landmarker.close()
            if self.hand_landmarker is not None:
                self.hand_landmarker.close()

        def detect_for_video(self, mp_image, timestamp_ms):
            pose_result = self.pose_landmarker.detect_for_video(mp_image, timestamp_ms)
            hand_result = self.hand_landmarker.detect_for_video(mp_image, timestamp_ms)

            pose_landmarks = pose_result.pose_landmarks[0] if pose_result.pose_landmarks else []
            left_hand_landmarks = []
            right_hand_landmarks = []

            for handedness, landmarks in zip(hand_result.handedness, hand_result.hand_landmarks):
                hand_label = handedness[0].category_name if handedness else None
                if hand_label == 'Left':
                    left_hand_landmarks = landmarks
                elif hand_label == 'Right':
                    right_hand_landmarks = landmarks

            return CompatHolisticResult([], pose_landmarks, left_hand_landmarks, right_hand_landmarks)

    print('Using MediaPipe Tasks HolisticLandmarker API')

def create_holistic_model():
    if USING_LEGACY_SOLUTIONS:
        return mp_holistic.Holistic(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )

    if os.path.exists(HOLISTIC_MODEL_PATH):
        base_options = python.BaseOptions(model_asset_path=HOLISTIC_MODEL_PATH)
        options = vision.HolisticLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_face_detection_confidence=0.5,
            min_face_landmarks_confidence=0.5,
            min_pose_detection_confidence=0.5,
            min_pose_landmarks_confidence=0.5,
            min_hand_landmarks_confidence=0.5,
        )
        return vision.HolisticLandmarker.create_from_options(options)

    if os.path.exists(POSE_MODEL_PATH) and os.path.exists(HAND_MODEL_PATH):
        print('holistic_landmarker.task not found, so the notebook will use pose + hand models. Face landmarks are disabled in this fallback mode.')
        return CombinedHolisticRunner(POSE_MODEL_PATH, HAND_MODEL_PATH)

    raise FileNotFoundError(
        "Latest MediaPipe installs use the Tasks API. Add 'holistic_landmarker.task' or keep both 'pose_landmarker.task' and 'hand_landmarker.task' next to this notebook."
    )

def mediapipe_detection(image, model, timestamp_ms=None):
    if USING_LEGACY_SOLUTIONS:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writable = False
        results = model.process(image)
        image.flags.writable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        return image, results

    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
    results = model.detect_for_video(mp_image, timestamp_ms)
    return image, results

def draw_landmarks(image, results):
    if USING_LEGACY_SOLUTIONS:
        mp_drawing.draw_landmarks(
            image, results.face_landmarks, mp_holistic.FACE_CONNECTIONS)
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
        mp_drawing.draw_landmarks(
            image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(
            image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        return

    mp_drawing.draw_landmarks(
        image, results.face_landmarks,
        vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION)
    mp_drawing.draw_landmarks(
        image, results.pose_landmarks,
        vision.PoseLandmarksConnections.POSE_LANDMARKS)
    mp_drawing.draw_landmarks(
        image, results.left_hand_landmarks,
        vision.HandLandmarksConnections.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(
        image, results.right_hand_landmarks,
        vision.HandLandmarksConnections.HAND_CONNECTIONS)

def draw_styled_landmarks(image, results):
    if USING_LEGACY_SOLUTIONS:
        mp_drawing.draw_landmarks(
            image, results.face_landmarks, mp_holistic.FACE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(80,255,121), thickness=1, circle_radius=1))
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2))
        mp_drawing.draw_landmarks(
            image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2))
        mp_drawing.draw_landmarks(
            image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
        return

    mp_drawing.draw_landmarks(
        image,
        results.face_landmarks,
        vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
        mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
        mp_drawing.DrawingSpec(color=(80,255,121), thickness=1, circle_radius=1))
    mp_drawing.draw_landmarks(
        image,
        results.pose_landmarks,
        vision.PoseLandmarksConnections.POSE_LANDMARKS,
        mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2))
    mp_drawing.draw_landmarks(
        image,
        results.left_hand_landmarks,
        vision.HandLandmarksConnections.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2))
    mp_drawing.draw_landmarks(
        image,
        results.right_hand_landmarks,
        vision.HandLandmarksConnections.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))

# Main function
cap = cv2.VideoCapture(0)

with create_holistic_model() as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        timestamp_ms = int(time.monotonic() * 1000)
        image, results = mediapipe_detection(frame, holistic, timestamp_ms)
        print(results)

        draw_styled_landmarks(image, results)
        cv2.imshow('OpenCV Feed', image)

        key = cv2.waitKey(10) & 0xFF
        if key == 27 or key == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Using MediaPipe Tasks HolisticLandmarker API
holistic_landmarker.task not found, so the notebook will use pose + hand models. Face landmarks are disabled in this fallback mode.
